Visualise model output from CF registry data vs baseline ppFEV1

In [1]:
import pandas as pd
import data.breathe_data as bd
import models.builders as mb
import src.models.helpers as mh
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import inference.helpers as ih
import numpy as np
import data.helpers as dh

In [34]:
df_meas = bd.load_meas_from_excel("CF_Registry_processed_with_idx", study_folder="CFT")
df_res = bd.load_meas_from_excel(
    # "infer_AR_using_fev1_CFT_10122025",
    "infer_AR_using_fev1_fef2575_CFT_10122025",
    study_folder="CFT",
    str_cols_to_arrays=["Airway resistance (%)"],
)
df_res = df_res.drop(columns=["Healthy FEV1 (L)"])

INFO:root:* Checking for same day measurements *
INFO:root:* Checking for same day measurements *


In [35]:
# Get AC from AR

AR = mh.VariableNode("Airway resistance (%)", 0, 90, 2, {"type": "uniform"})
AC = mh.VariableNode("Airway conductance (%)", 10, 100, 2, {"type": "uniform"})

df_res[AC.name] = df_res[AR.name].apply(lambda arr: arr[::-1])

# Check AC_mean = 100 - AR_mean
((df_res[AC.name].apply(lambda row: AC.get_mean(row)) - 100 + df_res[AR.name].apply(lambda row: AR.get_mean(row))) > 0.1).sum()

0

In [36]:
df = df_res.merge(df_meas, on=["ID", "Date Recorded"])

In [37]:
def get_dumbell_plot_data(df, ac_row):
    # Avoid modifying the original dataframe
    df_res = df.copy()

    # Assuming AC methods can take the row value directly
    mean = df_res[ac_row].apply(AC.get_mean)
    std = df_res[ac_row].apply(AC.get_std)
    
    df_res[f"{ac_row} mean"] = mean
    df_res[f"{ac_row} low"] = mean - std
    df_res[f"{ac_row} high"] = mean + std

    # Get Sorted IDs
    ids_sorted = df_res.sort_values("ecFEV1 % Predicted", ascending=False)["ID"].values

    # Prepare for Plotting
    # Map 'low' and 'high' to the same name ('dist') to group them using melt
    plot_cols = {
        f"{ac_row} low": f"{ac_row} dist",
        f"{ac_row} high": f"{ac_row} dist",
        f"{ac_row} mean": f"{ac_row} prediction",
    }

    df_melted = (
        df_res.rename(columns=plot_cols)
        .melt(
            id_vars=["ID"],
            value_vars=["ecFEV1 % Predicted", f"{ac_row} dist", f"{ac_row} prediction"],
            var_name="measure",
            value_name="value",
        )
        .set_index("ID")
        .loc[ids_sorted]
        .reset_index()
    )

    return df_melted, df_res, ids_sorted

In [38]:
df_to_plot, _, _ = get_dumbell_plot_data(df, AC.name)

In [39]:
# title = f"Dumbell plot for CF Registry data 2019, FEV1"
title = f"Dumbell plot for CF Registry data 2019, FEV1 & FEF25-75"
ac_col = AC.name

fig = make_subplots(
    1, 3, horizontal_spacing=0.1, column_titles=["Severe CF", "Moderate CF", "Mild CF"]
)

def plot_dumbell_for_df(fig, df, measures, col):
    # For measure[0]: 1 sigma up and down
    mask = df["measure"] == measures[0]
    for id in df[mask]["ID"].unique():
        mask_id = df["ID"] == id
        # Add mask
        mask_final = mask_id & mask
        fig.add_trace(
            go.Scatter(
                x=df[mask_final]["value"],
                y=df[mask_final]["ID"],
                mode="lines",
                name="ecFEV1 % healthy FEV1 * f(FEF25-75)",
                marker=dict(color="red"),
                line=dict(width=3),
                showlegend=(
                    True if id == df["ID"].unique()[0] else False
                )
            ),
            row=1,
            col=col,
        )
    # For measure[1]: clinical standard
    mask = df["measure"] == measures[1]
    ecfev1_prct_pred = df[mask]["value"]
    # Where above 100, set to 100
    ecfev1_prct_pred = np.clip(ecfev1_prct_pred, 0, 100)
    fig.add_trace(
        go.Scatter(
            x=ecfev1_prct_pred,
            y=df[mask]["ID"],
            mode="markers",
            name="ecFEV1%Predicted",
            marker=dict(size=4, color="blue"),
        ),
        row=1,
        col=col,
    )


# Split dataframe between mild, moderate and severe CF lung disease
# Equivalent to Mean AR_ecFEV1% < 30%, 30 to 60 and > 60%
# Get unique IDs and their corresponding Mean AR_ecFEV1% values
mask_ppfev1 = df_to_plot["measure"] == "ecFEV1 % Predicted"
# mask_ppfev1 = df_to_plot["measure"] == f"Mean {ac_col} prediction"

id_ppfev1 = df_to_plot[mask_ppfev1].groupby("ID")["value"].mean()

# Split IDs into three groups based on ecFEV1% values
mild_ids = id_ppfev1[id_ppfev1 >= 70].index
moderate_ids = id_ppfev1[
    (id_ppfev1 >= 40) & (id_ppfev1 < 70)
].index
severe_ids = id_ppfev1[id_ppfev1 < 40].index

# Create the three dataframes
df_mild = df_to_plot[df_to_plot["ID"].isin(mild_ids)]
df_moderate = df_to_plot[df_to_plot["ID"].isin(moderate_ids)]
df_severe = df_to_plot[df_to_plot["ID"].isin(severe_ids)]

title = f"{title} (#S {len(severe_ids)}, #M {len(moderate_ids)}, #M {len(mild_ids)})"

# Print the sizes to verify
print(f"# Mild: {len(mild_ids)}")
print(f"# Moderate: {len(moderate_ids)}")
print(f"# Severe: {len(severe_ids)}")

# Plot the three groups
plot_dumbell_for_df(fig, df_mild, [f"{ac_col} dist", "ecFEV1 % Predicted"], 3)
plot_dumbell_for_df(fig, df_moderate, [f"{ac_col} dist", "ecFEV1 % Predicted"], 2)
plot_dumbell_for_df(fig, df_severe, [f"{ac_col} dist", "ecFEV1 % Predicted"], 1)


fig.update_layout(
    height=1800,
    # height=1800,
    width=1200,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
print(min(df_to_plot["value"]))
fig.update_xaxes(
    range=[-25, 120],
    tickvals=[0, 40, 70, 100],
    title="Airway conductance (%)",
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.write_image(
    # f"{dh.get_path_to_main()}PlotsBreathe/Dumbell_plot_AR_ecFEV1_by_severity/{title}_clipped.pdf"
    f"{dh.get_path_to_main()}PlotsCFT/{title}_clipped.pdf"
)
fig.show()

# On this plot, if a person is very sick, it's best fev1 measurement will contain a lot of inflammatory markers (sputum, airway wall inflammation).
# The sicker the person,the more underestimated the AR pred is because the maximum FEV1 blown is not healthy.
# Let's add the FEF25-75.

# 60% of data falls within red band

# TODO: add vertical lines corresponding to key AR values

# Longitudinal AR profile on the web app

# Mild: 1055
# Moderate: 692
# Severe: 299
12.583652075440902
